In [1]:
import jax
import jax_sn
import jax.numpy as jnp
from jax_sn.domain import GeometricDomain, CrossSectionDataSingleGroup
from jax_sn.solution_domain import FEMMatrices, PrecomputedGeometricData
from jax_sn.operators.transport_operator import TransportOperatorVmapPallas, TransportOperatorBatchedPallas
from jax_sn.domain import TopologicalSortedAngle
import equinox as eqx
from jax_sn.tools.memory_analysis import _filter_jit_f_to_memory
%load_ext autoreload
%autoreload 2

# jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
# jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
# jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)

jax.config.update("jax_enable_x64", True)

In [2]:
dim =    3
degree = 1
resolution = 20
tn_order = 3
n_groups = 5
l_order = 4

jax.config.update("jax_log_compiles", False)
mgop = jax_sn.tools.setup_test_mgop.setup_slab_mgop(dim, degree, resolution, tn_order, n_groups, l_order, True, single_angle_transport_operator_cls = TransportOperatorBatchedPallas, single_angle_transport_operator_cls_kwargs = {"batch_size": 100})

In [3]:
mgop.n_elements

48000

In [4]:
xout = jnp.ones_like(mgop.transport_operator.out_structure())

In [5]:
from warp_kernel_v2 import WarpSweep
import numpy as onp
# top0 is a single-angle operator (defined earlier in the notebook)
sigma_t_jax = mgop.solution_domain_data.domain.cross_sections_indexed.map_to_full_array(0).total



warp_v2 = WarpSweep.from_transport_operator(mgop.transport_operator, sigma_t_jax)


Warp DeprecationWarning: The symbol `warp.vec` will soon be removed from the public API. Use `warp.types.vector` instead.
Warp 1.12.0 initialized:
   CUDA Toolkit 12.9, Driver 13.1
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce RTX 3080" (10 GiB, sm_86, mempool enabled)
   Kernel cache:
     /home/tbogaarts/.cache/warp/1.12.0


In [6]:
from warp_kernel_v2 import compute_inflow_and_omega_n_Je
import warp as wp
from jax_sn.operators.transport_operator.transport_operator_imp import _compute_omega_dot_n_Je_inflow
b = jnp.ones_like(mgop.transport_operator.out_structure())

result_warp = warp_v2.compute_all_angles(sigma_t_jax, b)

Module warp_kernel_v2 f8ba40a load on device 'cuda:0' took 23.54 ms  (cached)


In [7]:
%timeit warp_v2.compute_all_angles(sigma_t_jax, b)

630 ms ± 2.95 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [10]:
result_warp = warp_v2.compute_all_angles(sigma_t_jax, b)

In [11]:
%timeit jax_result = mgop.transport_operator.inv(b)

W0402 10:36:59.748672   12028 pjrt_executable.cc:638] Assume version compatibility. PjRt-IFRT does not track XLA executable versions.
W0402 10:36:59.808602   12028 pjrt_executable.cc:638] Assume version compatibility. PjRt-IFRT does not track XLA executable versions.


6.21 s ± 68 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
result_warp

array(shape=(72, 1296000), dtype=vec4d)

In [13]:
b = jnp.ones_like(mgop.transport_operator.out_structure())
result_jax = mgop.transport_operator.inv(b)
onp.testing.assert_allclose(result_jax, result_warp.numpy())

In [14]:
from warp_kernel_v2 import _compute_level_bounds, LevelBasedParallelismSetup


lsetup = LevelBasedParallelismSetup.from_transport_operator(mgop.transport_operator)

In [15]:
xs_wp = wp.from_jax(sigma_t_jax)


In [16]:
result_3 = warp_v2.multi_angle_levels(xs_wp, b, lsetup)

In [17]:
%timeit -n 10 -r 3 result_3 = wp.to_jax(warp_v2.multi_angle_levels(xs_wp, b, lsetup))

365 ms ± 8.92 ms per loop (mean ± std. dev. of 3 runs, 10 loops each)


In [20]:
result_3

array(shape=(72, 1296000), dtype=vec4d)

In [21]:
onp.testing.assert_allclose(result_3.numpy(), result_jax)